In [ ]:
%pip install --quiet matplotlib nltk pandas textblob

# Processing Text Data: From Messy Customer Feedback to Actionable Insight

In the previous topic, we scraped data from the web and structured it into a DataFrame. But what happens when the data we collect is *text*? Customer reviews, support tickets, social media posts — this is **unstructured data** in its purest form, and it rarely arrives in a state that's ready for analysis.

In this notebook, we'll work through a realistic text preprocessing pipeline using a dataset of customer feedback. By the end, you'll be able to:

- Clean and normalise raw text data
- Tokenise text into meaningful units
- Remove stop words and apply lemmatisation
- Build a reusable preprocessing function
- Extract basic insights through frequency analysis
- Perform a simple sentiment analysis (a gentle introduction to NLP)

Each of these steps forms part of a **preprocessing pipeline** — a deliberate sequence of transformations that turns messy, inconsistent text into something a machine (or a data scientist) can work with.

```
Raw Text → Cleaning → Tokenisation → Normalisation → Stop Word Removal → Lemmatisation → Analysis-Ready Text
```

Let's get started.

## Why does text need preprocessing?

In modules like Data Infrastructure and Data Engineering, you worked with highly structured data — rows and columns, consistent types, predictable formats. Text data is fundamentally different:

- People write inconsistently ("AMAZING", "amazing", "Amazing!!!")
- Text contains noise (URLs, email addresses, HTML tags, emoji)
- The same idea can be expressed many ways ("running", "ran", "runs")
- Common words like "the", "is", "and" dominate but carry little meaning

It's estimated that **80–90% of all data generated globally is unstructured**, and a large proportion of that is text. The ability to systematically clean and prepare text data is a foundational skill for any data scientist.

Let's see this in action with some customer feedback.

## Setting up our dataset

We'll work with a synthetic dataset of 50 customer reviews for an online retailer. These reviews are deliberately messy — they contain the kinds of inconsistencies you'd find in real-world feedback data.

In [ ]:
import pandas as pd

reviews = [
    {"id": 1, "review": "Absolutely LOVED this product!!! Best purchase I've made all year. 😊", "rating": 5},
    {"id": 2, "review": "terrible quality, fell apart after 2 days. DO NOT BUY. contact support@fakeshop.com for refund", "rating": 1},
    {"id": 3, "review": "It's okay I guess... nothing special but does the job. Delivery was quick tho", "rating": 3},
    {"id": 4, "review": "   Great value for money!  Fast shipping and well packaged.   ", "rating": 4},
    {"id": 5, "review": "WORST experience ever!!! Product arrived DAMAGED and customer service was unhelpful. See https://imgur.com/broken123 for pics", "rating": 1},
    {"id": 6, "review": "Really impressed with the build quality. Running smoothly after 6 months of daily use.", "rating": 5},
    {"id": 7, "review": "meh... its alright. wouldn't buy again but wouldn't return it either", "rating": 3},
    {"id": 8, "review": "Bought this as a gift for my daughter & she loves it!! Would definitely recommend to other parents 👍", "rating": 5},
    {"id": 9, "review": "The product is good BUT the delivery was absolutely terrible - 3 weeks late!!! Very disappointed with the service.", "rating": 2},
    {"id": 10, "review": "Five stars!!!!!!! Amazing product, amazing price, amazing everything!!!!!", "rating": 5},
    {"id": 11, "review": "Not what I expected based on the description. Colours are different from the photos online.", "rating": 2},
    {"id": 12, "review": "Works perfectly. Easy to set up - was running within 5 minutes of opening the box.", "rating": 4},
    {"id": 13, "review": "Received a USED product!!!!! clearly been opened before. Returning immediately. Avoid this seller.", "rating": 1},
    {"id": 14, "review": "Good product overall. Battery life could be better but everything else is spot on. ⭐⭐⭐⭐", "rating": 4},
    {"id": 15, "review": "i dont understand the hype... its just ok... nothing amazing about it tbh", "rating": 3},
    {"id": 16, "review": "<b>FANTASTIC!</b> Best thing I've bought in ages. Will be buying more as Christmas presents!", "rating": 5},
    {"id": 17, "review": "Stopped working after 1 week. Contacted support at help@fakeshop.com - no response after 5 days.", "rating": 1},
    {"id": 18, "review": "Pleasantly surprised by the quality at this price point. Feels much more premium than expected.", "rating": 4},
    {"id": 19, "review": "DO NOT WASTE YOUR MONEY!!! Cheap plastic rubbish that breaks easily. My kids were so disappointed 😢", "rating": 1},
    {"id": 20, "review": "Exactly as described. Good quality, fair price. Would shop here again.", "rating": 4},
    {"id": 21, "review": "The instructions were impossible to follow... took me 3 hours to assemble what should take 30 mins", "rating": 2},
    {"id": 22, "review": "LOVE LOVE LOVE this!!! Already ordered 2 more for friends. Check out my unboxing: https://youtube.com/watch?v=fake123", "rating": 5},
    {"id": 23, "review": "Average product. You get what you pay for I suppose. Nothing to write home about.", "rating": 3},
    {"id": 24, "review": "Brilliant customer service when I had an issue - replacement sent within 24 hours!", "rating": 5},
    {"id": 25, "review": "Product looks NOTHING like the pictures. Very misleading advertisement. Requesting a full refund.", "rating": 1},
    {"id": 26, "review": "My third purchase from this brand and they never disappoint. Consistently excellent quality.", "rating": 5},
    {"id": 27, "review": "Arrived with a small scratch on the surface but otherwise fine. Knocked off a star for packaging.", "rating": 3},
    {"id": 28, "review": "Absolutely not worth the premium price tag. Similar products available elsewhere for half the cost.", "rating": 2},
    {"id": 29, "review": "Kids love it, parents love it, everyone's happy! Perfect family product 🎉🎉🎉", "rating": 5},
    {"id": 30, "review": "Decent enough for the price. Some minor quality issues but nothing deal-breaking.", "rating": 3},
    {"id": 31, "review": "SCAM!!!! Never received my order and no response from seller. Reporting to trading standards.", "rating": 1},
    {"id": 32, "review": "Really well made. You can tell a lot of thought went into the design. Highly recommended.", "rating": 5},
    {"id": 33, "review": "Product is fine but shipping costs are RIDICULOUS. Almost as much as the item itself!!", "rating": 2},
    {"id": 34, "review": "Upgraded from the previous model and the improvements are noticeable. Great job by the manufacturer.", "rating": 4},
    {"id": 35, "review": "wouldn't recommend tbh... poor quality materials and flimsy construction. fell apart within a month", "rating": 1},
    {"id": 36, "review": "Excellent product!! Fast delivery, great packaging, and exactly what I needed. 10/10 👏", "rating": 5},
    {"id": 37, "review": "The colour is slightly different from what's shown online but otherwise very happy with purchase.", "rating": 4},
    {"id": 38, "review": "AVOID AVOID AVOID! Complete waste of money. Wish I'd read reviews before buying. Visit https://trustpilot.com/review/fakeshop for more complaints", "rating": 1},
    {"id": 39, "review": "Does exactly what it says on the tin. No complaints from me. Solid 4 stars.", "rating": 4},
    {"id": 40, "review": "Bought this on a whim and it's become one of my favourite things! Such a great find 😍", "rating": 5},
    {"id": 41, "review": "Instructions in broken English and missing parts. Had to order screws separately. Frustrating experience.", "rating": 2},
    {"id": 42, "review": "Super easy returns process when the first one arrived faulty. Replacement works perfectly!", "rating": 4},
    {"id": 43, "review": "Save your money... there are much better alternatives on the market at this price range", "rating": 2},
    {"id": 44, "review": "Outstanding quality! My partner and I both use it daily and it still looks brand new after 4 months.", "rating": 5},
    {"id": 45, "review": "   it's OK but nothing more...    would have liked better build quality for the price   ", "rating": 3},
    {"id": 46, "review": "Can't believe how good this is for the price!! Genuinely better than products costing 3x as much", "rating": 5},
    {"id": 47, "review": "Completely USELESS product. Broke on first use. Their returns policy is a joke - visit www.fakeshop.com/returns to see for yourself", "rating": 1},
    {"id": 48, "review": "Nice design, good functionality, reasonable price. Ticks all the boxes for me.", "rating": 4},
    {"id": 49, "review": "Second time ordering and quality has definitely declined since my first purchase last year. Disappointing.", "rating": 2},
    {"id": 50, "review": "Best. Purchase. Ever. Seriously, just buy it. You won't regret it!! 🙌", "rating": 5}
]

df = pd.DataFrame(reviews)

print(f"Dataset contains {len(df)} reviews")
df.head(10)

Take a moment to scan through the reviews above. You should be able to spot several problems that would make analysis difficult:

- Inconsistent casing ("LOVED", "loved", "Loved")
- Extra whitespace and padding
- URLs and email addresses
- HTML tags (`<b>FANTASTIC!</b>`)
- Emoji and special characters
- Excessive punctuation ("!!!!!")
- Informal language and abbreviations ("tho", "tbh")

Our goal is to systematically address these issues, step by step.

---

## Step 1: Text Cleaning

The first stage of any text preprocessing pipeline is **cleaning** — removing noise that doesn't contribute to the meaning of the text. We'll tackle this in layers, starting with the most straightforward operations.

### 1.1 Lowercasing

Converting all text to lowercase ensures that "Product", "PRODUCT", and "product" are all treated as the same word. Without this step, a frequency count would treat them as three separate terms.

In [ ]:
# Without lowercasing, these are treated as different words
words = ["Product", "PRODUCT", "product", "Product"]
print("Without normalisation:", set(words))

# With lowercasing, they're recognised as the same word
words_lower = [w.lower() for w in words]
print("With normalisation:   ", set(words_lower))

In [ ]:
# Apply lowercasing to our dataset
df["cleaned"] = df["review"].str.lower()

# Compare original and cleaned versions
df[["review", "cleaned"]].head(5)

### 1.2 Stripping whitespace

Extra spaces at the beginning and end of strings, or double-spaces within them, are common in user-generated text. These can cause problems when comparing or counting terms.

In [ ]:
# strip() removes leading and trailing whitespace
# We also replace multiple internal spaces with a single space
df["cleaned"] = df["cleaned"].str.strip()

# Replace multiple spaces with a single space
import re

df["cleaned"] = df["cleaned"].apply(lambda x: re.sub(r"\s+", " ", x))

# Check review 4 which had extra whitespace
print("Original: ", repr(df.loc[3, "review"]))
print("Cleaned:  ", repr(df.loc[3, "cleaned"]))

### 1.3 Removing URLs, email addresses, and HTML tags

Customer reviews sometimes contain URLs, email addresses, or even HTML markup. These are noise for text analysis purposes. We'll use **regular expressions** (regex) to find and remove them.

Regular expressions are a powerful pattern-matching language. We won't go deep into regex today, but it's worth understanding the basics — they're incredibly useful for text processing.

> **Further reading on regex:** The [Python `re` module documentation](https://docs.python.org/3/library/re.html) is an excellent reference, and [regex101.com](https://regex101.com/) is a brilliant interactive tool for building and testing patterns.

In [ ]:
def remove_noise(text: str) -> str:
    """
    Remove URLs, email addresses, and HTML tags from text.
    """
    # Remove URLs (http, https, www)
    text = re.sub(r"https?://\S+|www\.\S+", "", text)

    # Remove email addresses
    text = re.sub(r"\S+@\S+\.\S+", "", text)

    # Remove HTML tags
    text = re.sub(r"<.*?>", "", text)

    # Clean up any extra spaces left behind
    text = re.sub(r"\s+", " ", text).strip()

    return text


# Test on some examples
test_cases = [
    "Check out https://example.com for more info",
    "Email me at test@example.com please",
    "<b>bold text</b> and normal text",
]

for test in test_cases:
    print(f"Before: {test}")
    print(f"After:  {remove_noise(test)}")
    print()

In [ ]:
# Apply to our dataset
df["cleaned"] = df["cleaned"].apply(remove_noise)

# Check reviews that had URLs/emails (reviews 2, 5, 16)
for idx in [1, 4, 15]:
    print(f"Review {idx + 1}:")
    print(f"  Original: {df.loc[idx, 'review']}")
    print(f"  Cleaned:  {df.loc[idx, 'cleaned']}")
    print()

### 1.4 Removing punctuation and special characters

Excessive punctuation ("!!!!!!") and special characters add noise. We'll remove punctuation but keep alphanumeric characters and spaces.

In [ ]:
def remove_punctuation(text: str) -> str:
    """
    Remove punctuation and special characters, keeping only
    letters, numbers, and spaces.
    """
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


df["cleaned"] = df["cleaned"].apply(remove_punctuation)

# Check a few examples
for idx in [0, 9, 28]:
    print(f"Review {idx + 1}:")
    print(f"  Original: {df.loc[idx, 'review']}")
    print(f"  Cleaned:  {df.loc[idx, 'cleaned']}")
    print()

Our text is now much cleaner. Let's look at the full dataset so far.

In [ ]:
df[["review", "cleaned"]].head(10)

---

## Step 2: Tokenisation

**Tokenisation** is the process of splitting text into individual units (tokens). Usually, this means splitting into words, but you can also tokenise into sentences.

Why not just use Python's built-in `.split()` method? It works for simple cases, but dedicated tokenisers handle edge cases better — contractions, hyphenated words, and punctuation boundaries.

We'll use NLTK (Natural Language Toolkit), one of the most widely-used NLP libraries in Python.

In [ ]:
import nltk

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

from nltk.tokenize import word_tokenize, sent_tokenize

In [ ]:
# Compare Python's split() with NLTK's word_tokenize()
example = "I've been running daily and it's been life-changing"

print("str.split():     ", example.split())
print("word_tokenize(): ", word_tokenize(example))

Notice how `word_tokenize` handles the contraction "I've" by splitting it into "I" and "'ve", and treats "it's" similarly. This is more linguistically accurate — these are separate meaningful units.

Since we've already removed punctuation from our cleaned text, the difference will be less pronounced here, but using a proper tokeniser is good practice.

In [ ]:
# Tokenise our cleaned reviews
df["tokens"] = df["cleaned"].apply(word_tokenize)

# Let's see the tokens for the first few reviews
for idx in range(5):
    print(f"Review {idx + 1}: {df.loc[idx, 'tokens']}")
    print()

In [ ]:
# Sentence tokenisation is also useful - let's see it on a longer review
example_review = df.loc[8, "review"]

print("Original review:")
print(example_review)
print("\nSentences:")
for i, sentence in enumerate(sent_tokenize(example_review), 1):
    print(f"  {i}. {sentence}")

---

## Step 3: Stop Word Removal

**Stop words** are common words that appear frequently in text but typically carry little meaning on their own — words like "the", "is", "and", "a", "to". Removing them helps us focus on the words that actually convey the content of the review.

NLTK provides a built-in list of English stop words.

In [ ]:
nltk.download("stopwords", quiet=True)

from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

print(f"NLTK has {len(stop_words)} English stop words.")
print(f"\nSome examples: {sorted(list(stop_words))[:20]}")

### ⚠️ A word of caution

Stop word removal isn't always straightforward. Look at the NLTK stop words list above — you might notice that words like **"not"**, **"no"**, **"don't"**, and **"wouldn't"** are included.

For general frequency analysis, removing these is fine. But for **sentiment analysis** (which we'll touch on later), removing negation words can completely flip the meaning of a sentence:

- "I would **not** recommend this product" → "I would recommend this product"

This is something to keep in mind as you develop your own pipelines. The "right" preprocessing steps depend on *what you plan to do with the text afterwards*.

In [ ]:
# Remove stop words from our tokens
df["tokens_no_stop"] = df["tokens"].apply(
    lambda tokens: [word for word in tokens if word not in stop_words]
)

# Compare before and after for a few reviews
for idx in [0, 2, 6]:
    print(f"Review {idx + 1}:")
    print(f"  Before ({len(df.loc[idx, 'tokens'])} tokens): {df.loc[idx, 'tokens']}")
    print(f"  After  ({len(df.loc[idx, 'tokens_no_stop'])} tokens): {df.loc[idx, 'tokens_no_stop']}")
    print()

Notice how much shorter the token lists become. The remaining words — "loved", "product", "best", "purchase" — are much more informative about what the customer actually said.

---

## Step 4: Stemming and Lemmatisation

Different forms of the same word — "running", "runs", "ran" — should ideally be treated as one term. There are two main approaches to this:

**Stemming** chops off word endings using simple rules to find a "stem". It's fast but crude — the results aren't always real words.

**Lemmatisation** uses a dictionary (a *lemma* is the dictionary form of a word) to find the proper root form. It's slower but produces more meaningful results.

Let's compare them.

In [ ]:
nltk.download("wordnet", quiet=True)

from nltk.stem import PorterStemmer, WordNetLemmatizer

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# Compare stemming and lemmatisation on various word forms
test_words = [
    "running", "runs", "ran",
    "better", "best",
    "easily", "easier",
    "disappointed", "disappointing", "disappointment",
    "recommended", "recommending", "recommendation",
    "quality", "qualities",
]

print(f"{'Word':<20} {'Stem':<20} {'Lemma':<20}")
print("-" * 60)
for word in test_words:
    stem = stemmer.stem(word)
    lemma = lemmatizer.lemmatize(word)
    print(f"{word:<20} {stem:<20} {lemma:<20}")

Notice some key differences:

- The stemmer produces non-words like "easili" and "disappoint" from "easily" and "disappointed"
- The lemmatiser is more conservative — it sometimes doesn't reduce the word ("better" stays as "better" because the default part-of-speech is noun)
- The stemmer is more aggressive at grouping variants together

For most text analysis tasks, **lemmatisation is preferred** because the output is more interpretable. The assessment rubric for this module specifically mentions lemmatisation as an important preprocessing step — implementing it properly is one of the things that distinguishes a low pass from a pass.

We'll use lemmatisation for our pipeline.

In [ ]:
# Apply lemmatisation to our filtered tokens
df["tokens_lemmatised"] = df["tokens_no_stop"].apply(
    lambda tokens: [lemmatizer.lemmatize(word) for word in tokens]
)

# Compare the pipeline so far for a few reviews
for idx in [5, 8, 11]:
    print(f"Review {idx + 1}:")
    print(f"  Original:    {df.loc[idx, 'review']}")
    print(f"  Cleaned:     {df.loc[idx, 'cleaned']}")
    print(f"  Tokens:      {df.loc[idx, 'tokens_no_stop']}")
    print(f"  Lemmatised:  {df.loc[idx, 'tokens_lemmatised']}")
    print()

---

## Building a Reusable Preprocessing Function

So far, we've applied each step individually. In practice, you'll want to combine them into a single, reusable function. This is good software engineering practice and will serve you well in your summative assessment — your automated process should be clean, modular, and reproducible.

Let's bring everything together into one function.

In [ ]:
def preprocess_text(text: str) -> list[str]:
    """
    Apply a full text preprocessing pipeline to a string.

    Steps:
        1. Convert to lowercase
        2. Strip extra whitespace
        3. Remove URLs, email addresses, and HTML tags
        4. Remove punctuation and special characters
        5. Tokenise
        6. Remove stop words
        7. Lemmatise

    Args:
        text: The raw text string to preprocess.

    Returns:
        A list of cleaned, lemmatised tokens.
    """
    stop_words = set(stopwords.words("english"))
    lemmatizer = WordNetLemmatizer()

    # Step 1: Lowercase
    text = text.lower()

    # Step 2: Strip whitespace
    text = re.sub(r"\s+", " ", text).strip()

    # Step 3: Remove URLs, emails, HTML
    text = re.sub(r"https?://\S+|www\.\S+", "", text)
    text = re.sub(r"\S+@\S+\.\S+", "", text)
    text = re.sub(r"<.*?>", "", text)

    # Step 4: Remove punctuation
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()

    # Step 5: Tokenise
    tokens = word_tokenize(text)

    # Step 6: Remove stop words
    tokens = [word for word in tokens if word not in stop_words]

    # Step 7: Lemmatise
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return tokens

In [ ]:
# Test our pipeline function on a messy review
messy_review = "WORST experience ever!!! Product arrived DAMAGED and customer service was unhelpful. See https://imgur.com/broken123 for pics"

print("Input: ", messy_review)
print("Output:", preprocess_text(messy_review))

In [ ]:
# Apply the full pipeline to the entire dataset
df["processed_tokens"] = df["review"].apply(preprocess_text)

# Also create a joined string version (useful for some analyses)
df["processed_text"] = df["processed_tokens"].apply(lambda tokens: " ".join(tokens))

df[["review", "processed_text"]].head(10)

---

## Step 5: Frequency Analysis

Now that our text is clean and preprocessed, we can start extracting insights. The simplest form of text analysis is **word frequency counting** — which words appear most often across all reviews?

Python's `collections.Counter` is ideal for this.

In [ ]:
from collections import Counter

# Flatten all tokens into a single list
all_tokens = [token for tokens in df["processed_tokens"] for token in tokens]

print(f"Total tokens across all reviews: {len(all_tokens)}")
print(f"Unique tokens: {len(set(all_tokens))}")

# Count the most common words
word_counts = Counter(all_tokens)
print("\nTop 15 most common words:")
for word, count in word_counts.most_common(15):
    print(f"  {word:<20} {count}")

Let's visualise this as a bar chart. A clear, well-labelled visualisation is much more effective than a raw frequency table — and it's the kind of output that will strengthen your summative work.

In [ ]:
import matplotlib.pyplot as plt

# Get the top 20 words
top_words = word_counts.most_common(20)
words, counts = zip(*top_words)

fig, ax = plt.subplots(figsize=(12, 6))

ax.barh(range(len(words)), counts, color="steelblue")
ax.set_yticks(range(len(words)))
ax.set_yticklabels(words)
ax.invert_yaxis()
ax.set_xlabel("Frequency")
ax.set_title("Top 20 Most Frequent Words in Customer Reviews")

plt.tight_layout()
plt.show()

### Frequency analysis by rating

We can go further by comparing word frequencies across different rating groups. What do happy customers talk about versus unhappy ones?

In [ ]:
# Split reviews into positive (4-5 stars) and negative (1-2 stars)
positive_tokens = [
    token
    for tokens, rating in zip(df["processed_tokens"], df["rating"])
    if rating >= 4
    for token in tokens
]

negative_tokens = [
    token
    for tokens, rating in zip(df["processed_tokens"], df["rating"])
    if rating <= 2
    for token in tokens
]

positive_counts = Counter(positive_tokens)
negative_counts = Counter(negative_tokens)

# Display side by side
print(f"{'Positive (4-5 ⭐)':<30} {'Negative (1-2 ⭐)':<30}")
print("-" * 60)

pos_top = positive_counts.most_common(10)
neg_top = negative_counts.most_common(10)

for i in range(10):
    pos = f"{pos_top[i][0]} ({pos_top[i][1]})" if i < len(pos_top) else ""
    neg = f"{neg_top[i][0]} ({neg_top[i][1]})" if i < len(neg_top) else ""
    print(f"  {pos:<28} {neg:<28}")

This is already useful — you can start to see patterns in how positive and negative reviewers use different language. But frequency analysis has limitations: it treats each word independently and doesn't capture meaning, context, or the relationships between words.

This is where **Natural Language Processing (NLP)** comes in. In later topics, we'll explore NLP techniques in depth — including TF-IDF, topic modelling, and named entity recognition. For now, let's get a taste of what's possible with a simple sentiment analysis.

---

## Step 6: Introduction to Sentiment Analysis

**Sentiment analysis** is the process of determining the emotional tone behind text — is a review positive, negative, or neutral? It's one of the most common applications of NLP and is widely used in business to understand customer opinion at scale.

We'll use **TextBlob**, a beginner-friendly library that provides a simple API for common NLP tasks. TextBlob's sentiment analyser returns two values:

- **Polarity**: A score from -1.0 (very negative) to +1.0 (very positive)
- **Subjectivity**: A score from 0.0 (very objective) to 1.0 (very subjective)

We'll explore this more fully in later sessions — for now, the goal is simply to see how preprocessing feeds into analysis.

In [ ]:
from textblob import TextBlob

In [ ]:
# Let's see how TextBlob scores a few example sentences
examples = [
    "This product is absolutely wonderful and I love it",
    "Terrible quality, complete waste of money",
    "The product arrived on Tuesday",
    "It's okay I guess",
]

for text in examples:
    blob = TextBlob(text)
    print(f"Text:        {text}")
    print(f"Polarity:    {blob.sentiment.polarity:+.2f}")
    print(f"Subjectivity: {blob.sentiment.subjectivity:.2f}")
    print()

### Preprocessing vs. raw text for sentiment analysis

An important question: should we run sentiment analysis on the **raw text** or the **preprocessed text**? Let's compare and find out.

This is a great example of how the "right" preprocessing depends on your downstream task. Some sentiment tools actually *use* capitalisation ("AMAZING" is stronger than "amazing") and punctuation ("!!!" signals intensity) as signals.

In [ ]:
# Calculate sentiment on both raw and preprocessed text
df["sentiment_raw"] = df["review"].apply(
    lambda x: TextBlob(x).sentiment.polarity
)
df["sentiment_processed"] = df["processed_text"].apply(
    lambda x: TextBlob(x).sentiment.polarity
)

# Compare the results
comparison = df[["review", "rating", "sentiment_raw", "sentiment_processed"]].copy()
comparison["difference"] = abs(
    comparison["sentiment_raw"] - comparison["sentiment_processed"]
)

# Show cases where preprocessing changed the sentiment score most
print("Reviews where preprocessing had the biggest impact on sentiment:\n")
top_diffs = comparison.nlargest(5, "difference")

for _, row in top_diffs.iterrows():
    print(f"  Review: {row['review'][:80]}...")
    print(f"  Rating: {row['rating']} ⭐  |  Raw sentiment: {row['sentiment_raw']:+.2f}  |  Processed: {row['sentiment_processed']:+.2f}")
    print()

### Visualising sentiment

Let's visualise the distribution of sentiment scores and see how well they align with the star ratings customers gave.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of sentiment scores
axes[0].hist(df["sentiment_raw"], bins=20, color="steelblue", edgecolor="white", alpha=0.8)
axes[0].set_xlabel("Sentiment Polarity")
axes[0].set_ylabel("Number of Reviews")
axes[0].set_title("Distribution of Sentiment Scores")
axes[0].axvline(x=0, color="red", linestyle="--", alpha=0.5, label="Neutral")
axes[0].legend()

# Sentiment vs star rating
colours = {1: "#d62728", 2: "#ff7f0e", 3: "#bcbd22", 4: "#2ca02c", 5: "#1f77b4"}
for rating in sorted(df["rating"].unique()):
    subset = df[df["rating"] == rating]
    axes[1].scatter(
        subset["rating"],
        subset["sentiment_raw"],
        c=colours[rating],
        label=f"{rating} ⭐",
        alpha=0.6,
        s=60,
    )

axes[1].set_xlabel("Star Rating")
axes[1].set_ylabel("Sentiment Polarity")
axes[1].set_title("Sentiment Score vs. Star Rating")
axes[1].axhline(y=0, color="red", linestyle="--", alpha=0.5)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Average sentiment by rating
avg_sentiment = df.groupby("rating")["sentiment_raw"].mean()

print("Average sentiment polarity by star rating:\n")
for rating, sentiment in avg_sentiment.items():
    bar = "█" * int((sentiment + 1) * 20)  # Simple visual bar
    print(f"  {rating} ⭐: {sentiment:+.3f}  {bar}")

### What can we observe?

There should be a general trend: higher star ratings tend to correspond with more positive sentiment scores. But it's far from perfect — some 1-star reviews might get positive sentiment scores (perhaps because the customer says "I *wanted* to love this product"), and some 5-star reviews might score lower than expected.

This is a key insight: **sentiment analysis is useful but imperfect**. The simple lexicon-based approach used by TextBlob doesn't understand context, sarcasm, or nuance. In later sessions, we'll explore more sophisticated NLP techniques that handle these challenges better.

---

## Summary

In this notebook, we've built a complete text preprocessing pipeline and used it to analyse customer feedback. Here's what we covered:

| Step | Purpose | Key tools |
|---|---|---|
| Cleaning | Remove noise (URLs, HTML, extra spaces) | `re` (regex), string methods |
| Tokenisation | Split text into meaningful units | `nltk.word_tokenize` |
| Stop word removal | Filter out common, low-information words | `nltk.corpus.stopwords` |
| Lemmatisation | Reduce words to their dictionary form | `nltk.WordNetLemmatizer` |
| Frequency analysis | Count and visualise common terms | `collections.Counter`, `matplotlib` |
| Sentiment analysis | Determine emotional tone of text | `TextBlob` |

### Key takeaways

1. **Preprocessing is not one-size-fits-all.** The steps you choose depend on your downstream task. Sentiment analysis tools may work *better* on raw text because they use capitalisation and punctuation as signals.

2. **Build reusable functions.** Wrapping your pipeline in a function makes your code reproducible and easier to test — both of which are important for your summative assessment.

3. **Lemmatisation over stemming.** Lemmatisation produces more interpretable results and is generally preferred in professional text analysis work.

4. **Simple tools can be powerful.** Even basic frequency analysis reveals patterns in how customers express satisfaction and dissatisfaction.

---

## Exercises

Try these to consolidate your understanding:

1. **Customise the stop words list.** Add domain-specific stop words (e.g., "product", "buy", "order") that appear frequently but don't help distinguish between positive and negative reviews. How does this affect your frequency analysis?

2. **Try stemming instead of lemmatisation.** Replace the lemmatiser with `PorterStemmer` in the `preprocess_text` function. Compare the output — which produces more readable results?

3. **Analyse 3-star reviews separately.** These "neutral" reviews often contain the most constructive feedback. What words appear frequently in 3-star reviews but not in 5-star or 1-star reviews?

4. **Extend the pipeline.** Add a step to handle common abbreviations ("tbh" → "to be honest", "tho" → "though"). Where in the pipeline should this step go, and why?

---

## What's next?

In later topics, we'll build on these foundations to explore more advanced NLP techniques — including TF-IDF (a smarter approach to measuring word importance), topic modelling, and named entity recognition. The preprocessing pipeline you've built today is the starting point for all of that work.